# House Prices Modeling - Data Science in Production (PW1)

This notebook builds a modeling pipeline for predicting house prices using the Kaggle House Prices dataset.

Steps included:
- Dataset loading
- Feature selection
- Feature processing (scaling and encoding)
- Model training
- Model evaluation using RMSLE

In [12]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_log_error

## 1. Dataset Loading

Load the training and test datasets from the local data folder.

In [13]:
train = pd.read_csv("../data/train.csv")
test = pd.read_csv("../data/test.csv")

train.head()

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


In [14]:
print("Train shape:", train.shape)
print("Test shape:", test.shape)

Train shape: (1460, 81)
Test shape: (1459, 80)


## 2. Feature Selection

Select at least two continuous and two categorical features for modeling.

In [15]:
features = ['GrLivArea', 'LotArea', 'Neighborhood', 'HouseStyle']
target = 'SalePrice'

data = train[features + [target]]

data.head()

,GrLivArea,LotArea,Neighborhood,HouseStyle,SalePrice
0,1710,8450,CollgCr,2Story,208500
1,1262,9600,Veenker,1Story,181500
2,1786,11250,CollgCr,2Story,223500
3,1717,9550,Crawfor,2Story,140000
4,2198,14260,NoRidge,2Story,250000


## 3. Train-Test Split

Split the dataset into training and testing sets.

In [16]:
X = data[features]
y = data[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training samples:", X_train.shape)
print("Testing samples:", X_test.shape)

Training samples: (1168, 4)
Testing samples: (292, 4)


## 4. Feature Processing

Continuous features are scaled and categorical features are encoded.

In [17]:
continuous_features = ['GrLivArea', 'LotArea']

scaler = StandardScaler()

X_train_cont = scaler.fit_transform(X_train[continuous_features])
X_test_cont = scaler.transform(X_test[continuous_features])

In [18]:
categorical_features = ['Neighborhood', 'HouseStyle']

encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

X_train_cat = encoder.fit_transform(X_train[categorical_features])
X_test_cat = encoder.transform(X_test[categorical_features])

In [19]:
X_train_processed = np.hstack((X_train_cont, X_train_cat))
X_test_processed = np.hstack((X_test_cont, X_test_cat))

print("Processed training shape:", X_train_processed.shape)
print("Processed testing shape:", X_test_processed.shape)

Processed training shape: (1168, 35)
Processed testing shape: (292, 35)


## 5. Model Training

Train a Linear Regression model on the processed features.

In [20]:
model = LinearRegression()

model.fit(X_train_processed, y_train)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for the `lsqr` solver.`tol` is set as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. This parameter has no effect when fittingon dense data... versionadded:: 1.7",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False


In [21]:
y_pred = model.predict(X_test_processed)

# Ensure predictions are positive
y_pred = np.maximum(y_pred, 1)

y_pred[:10]

array([125976.51580425, 333459.35233815, 107895.71043949, 152981.96665293,
       229376.9146021 ,  82388.5748897 , 219216.8276659 , 149229.41765446,
        82591.17845063, 117694.79343644])

## 6. Model Evaluation

Evaluate the model using RMSLE (Root Mean Squared Log Error), the competition metric.

In [22]:
def compute_rmsle(y_test: np.ndarray, y_pred: np.ndarray, precision: int = 2) -> float:
    """Note: y_pred values must be positive (log is applied)."""
    rmsle = np.sqrt(mean_squared_log_error(y_test, y_pred))
    return round(rmsle, precision)

score = compute_rmsle(y_test, y_pred)

print("RMSLE:", score)

RMSLE: 0.2
